In [6]:
from config import settings
from utils.prompts import Prompts

import json
from openai import OpenAI
from typing import Dict, List, Optional



base_url = settings.OPEN_ROUTER_BASE_URL
api_key = settings.LLM_KEY

openai_client = OpenAI(api_key=api_key, base_url=base_url)
openai_model = "openai/gpt-oss-120b"

In [12]:
def llm(prompt, model):
    response = openai_client.responses.create(
        model=model,
        input=prompt
    )
    return response

In [14]:
new_response = llm("Hello, how are you?", openai_model)
type(new_response.output)

list

In [15]:
type(new_response.output_text)

str

In [17]:
def generate_recommendations(
        job_data: Optional[List[Dict]], llm_client: OpenAI, instructions: str, model: str = "openai/gpt-oss-120b"
) -> Optional[List[Dict]]:
    """
    Generate portfolio recommendations for a job using LLM.
    :param job_data: List of job data (dictionaries).
    :param llm_client: The LLM client responsible for generating recommendations.
    :param instructions: System instructions for the LLM.
    :param model: llm model to use
    :return: A list of jobs recommendations.
    """
    try:
        message=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": job_data},
        ]

        response = llm_client.responses.create(
            model=model,
            input=message,
            text={
                "format": {
                    "type": "json_object"
                }
            }
        )
        recommendations = response.output_text
        return json.loads(recommendations)
    except Exception as e:
        return {
            "success": False,
            "error": {
                "type": str(type(e).__name__),
                "message": str(e)
            }
        }


